# Model Evaluation and Deployment

This notebook evaluates the final model against a benchmark heuristic and deploys the model to SageMaker.

In [ ]:
import joblib
import pandas as pd
from sklearn.metrics import classification_report

# Load combined test data
test_df = pd.read_csv("split/Xy_test.csv")

# Adjust this column name as needed
label_column = "label"  # Replace with your actual label column name

# Split into features and labels
X_test = test_df.drop(columns=[label_column, "record_id"])
y_test = test_df[label_column]

# Load trained model
model = joblib.load("model-output/final_model.pkl")

# Predict
y_pred = model.predict(X_test)
print("Final Model Performance:\n")
print(classification_report(y_test, y_pred))

In [ ]:
# Load heuristic predictions (ensure this CSV aligns with y_test order)
heuristic_preds = pd.read_csv("data/heuristic_predictions.csv").squeeze()

print("Heuristic Model Performance:\n")
print(classification_report(y_test, heuristic_preds))

## Comparison Summary

Use the above reports to compare accuracy, precision, recall, and F1-score between your trained model and the heuristic baseline.

In [19]:
import sagemaker
from sagemaker.xgboost.model import XGBoostModel
from sagemaker.model_monitor import DataCaptureConfig
from sagemaker import get_execution_role
import sys
import time

sys.path.append('../config')
import config

# SageMaker setup
sagemaker_session = sagemaker.Session()
role = get_execution_role()
bucket = config.S3_BUCKET

# Update with your actual model S3 path and entry point script
model_artifact = f's3://{bucket}/{config.MODEL_ARTIFACT_PATH}'
s3_data_capture_path = f's3://{bucket}/final_project/monitoring/data-capture'

# Create and deploy model
xgb_model = XGBoostModel(
    model_data=model_artifact,
    role=role,
    framework_version="1.5-1",
    sagemaker_session=sagemaker_session
)

endpoint_name = f"final-model-endpoint-{int(time.time())}"
predictor = xgb_model.deploy(
    initial_instance_count=1,
    instance_type='ml.m5.large',
    endpoint_name=endpoint_name,
    data_capture_config=DataCaptureConfig(
        enable_capture=True,
        destination_s3_uri=s3_data_capture_path,
        csv_content_types=['text/csv']
    )
)

print(f"Model deployed. Endpoint name: {endpoint_name}")

------!Model deployed. Endpoint name: final-model-endpoint-1750522664


In [31]:
from sagemaker.serializers import CSVSerializer
import numpy as np

predictor.serializer = CSVSerializer()
sample_input = X_test.iloc[:5]
results_raw = predictor.predict(sample_input.values)
predictions_prob = [float(result[0]) for result in results_raw]
predictions_binary = (np.array(predictions_prob) > 0.5).astype(int)

print("Raw probabilities from endpoint:", predictions_prob)
print("Binary predictions (threshold > 0.5):", predictions_binary.tolist())
print("Actual labels for this sample:", y_test.iloc[:5].values.tolist())

Raw probabilities from endpoint: [5.019500349590089e-06, 0.9999912977218628, 0.000156313632032834, 0.9999396800994873, 5.019500349590089e-06]
Binary predictions (threshold > 0.5): [0, 1, 0, 1, 0]
Actual labels for this sample: [0, 1, 0, 1, 0]
